# Notebook 28 — Nested Random Forest Optimization

**Research objective:** Focus exclusively on Random Forest. Compare price vs. price+volume, leaf size, depth, and class balancing, with **inner-only** model/threshold selection and three chronological outer evaluations. No locked historical holdout access, no prospective-ledger writes.

**Interpretation:** Accuracy and entry policies are separately selected. Thresholds and model choices are selected using only inner folds. Estimated net returns subtract a flat 10 bps per selected entry; they are **not** a portfolio backtest. Correlated ticker returns, slippage, capacity, overlapping trades, and multiple testing can make estimates optimistic.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
if not (ROOT/'src').exists():raise RuntimeError('Open from repository root or notebooks directory')
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.rf_optimization import run_rf_optimization
import pandas as pd
import numpy as np
print('Project:',ROOT)


## 1. Experiment configuration

5 RF settings × 2 feature families × 2 inner folds × 3 outer folds = **60 inner model fits**, plus **6 outer fits** for separately selected accuracy and entry policies. Each inner fit evaluates 5 thresholds without refitting.

In [ ]:
N_OUTER=3
N_INNER=2
TRAIN_CAP=35000
N_ESTIMATORS=120
COST_BPS=10
MIN_ENTRIES=100
MIN_COVERAGE=.02
OUTPUT=ROOT/'artifacts'/'rf_optimization'/'1d'
print('Inner fits:',N_OUTER*N_INNER*5*2,'Outer fits:',N_OUTER*2)


## 2. Load the existing daily master

Unpack the complete four-value `load_master()` return. Notebook 28 does not regenerate the master or access the historical holdout for scoring.

In [ ]:
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
paths=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
source=next((p for p in paths if all((p/f).exists() for f in required)),None)
if source is None:raise FileNotFoundError('Missing daily master artifacts; run Notebook 03')
master,latest,bundle,manifest=load_master(source)
assert isinstance(master,pd.DataFrame),type(master)
print('Rows:',len(master),'Symbols:',master.symbol.nunique(),'Range:',master.bar_timestamp.min(),master.bar_timestamp.max())


## 3. Run nested Random Forest research

This is intentionally computationally intensive. Training uses a deterministic row cap, and each inner fold refits target quantiles and base transformations using only its own training history.

In [ ]:
results,development_cutoff=run_rf_optimization(master,n_outer=N_OUTER,n_inner=N_INNER,
    train_cap=TRAIN_CAP,n_estimators=N_ESTIMATORS,cost_bps=COST_BPS,
    min_entries=MIN_ENTRIES,min_coverage=MIN_COVERAGE,output_dir=OUTPUT)
print('Development cutoff:',development_cutoff)
print('CSV outputs:',OUTPUT)


## 4. Outer-fold comparison and policy decisions

In [ ]:
outer=results['outer']
display(outer.round(5))
display(outer.groupby('objective').agg(accuracy=('accuracy','mean'),auc=('auc','mean'),
    baseline=('majority_baseline','mean'),precision=('precision','mean'),coverage=('coverage','mean'),
    mean_net=('mean_net','mean'),entry_count=('entry_count','sum')).round(5))
print('Models selected using inner folds only:')
display(results['selections'][['fold','objective','config','features','threshold','accuracy','precision','coverage','min_entries']])


## 5. Inner-fold model and feature ablations

In [ ]:
inner=results['inner']
display(inner.groupby(['config','features']).agg(accuracy=('accuracy','mean'),auc=('auc','mean'),
    precision=('precision','mean'),coverage=('coverage','mean')).sort_values('accuracy',ascending=False).round(5))


## 6. Trade distributions and stability

Only out-of-sample selected trades are shown; do not infer independent observations from correlated ticker bars.

In [ ]:
trades=results['trades']
if len(trades):
    display(trades.groupby(['objective','fold']).agg(n=('net_return','size'),
        mean_net=('net_return','mean'),median_net=('net_return','median'),
        win_rate=('net_return',lambda x:(x>0).mean()),
        p05=('net_return',lambda x:x.quantile(.05)),
        p95=('net_return',lambda x:x.quantile(.95))).round(5))
    display(trades.groupby(['objective','symbol']).agg(n=('net_return','size'),mean_net=('net_return','mean'))
            .sort_values(['objective','n'],ascending=[True,False]).head(30).round(5))
else:print('No trades passed the inner-selected policies')


## 7. Feature importance and runtime

In [ ]:
display(results['importances'].groupby(['objective','feature']).importance.mean()
        .sort_values(ascending=False).groupby(level=0).head(20))
display(results['timings'])


## 8. Research gate

The gate is intentionally conservative: a strategy is **not validated** merely because it has positive average net returns. At least three positive outer folds, adequate coverage, and a positive precision lift are minimum screening conditions, not proof of profitability.

In [ ]:
for objective,group in outer.groupby('objective'):
    eligible=(len(group)==3 and (group.mean_net>0).all() and
              (group.precision>group.up_base).all() and
              (group.coverage>=MIN_COVERAGE).all() and
              (group.entry_count>=MIN_ENTRIES).all())
    print(objective,':','RESEARCH CANDIDATE — needs independent prospective validation' if eligible else 'NO PROMOTION — research only')
print('Historical holdout and prospective ledger unchanged.')
